In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoConfig

from pytorch_lightning import LightningModule, Trainer

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import precision_score, recall_score, f1_score

# Dummy dataset - replace with your own
data = [
    ("I love this movie!", 1),
    ("Terrible plot, I hated it.", 0),
    ("Pretty good overall.", 1),
    ("Worst acting ever.", 0),
]

class TextClassificationDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, label = self.data[idx]
        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

dataset = TextClassificationDataset(data, tokenizer)
loader = DataLoader(dataset, batch_size=2, shuffle=True)


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


### Model Setup

In [2]:
class LitTextClassifier(LightningModule):
    def __init__(self, model_name="bert-base-uncased", n_classes=2, lr=2e-5):
        super().__init__()
        self.save_hyperparameters() # NOTE: important to track hyperparams in MLflow

        # Load model and config dynamically
        self.config = AutoConfig.from_pretrained(model_name)
        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        self.classifier = nn.Linear(self.config.hidden_size, n_classes)
        self.lr = lr

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output  # [batch_size, hidden_size]
        return self.classifier(pooled_output)

    def training_step(self, batch, batch_idx): # NOTE: this has to be defined
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["label"]

        logits = self(input_ids, attention_mask)
        loss = F.cross_entropy(logits, labels)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == labels).float().mean()
        precision = precision_score(labels.cpu(), preds.cpu(), average="binary")
        recall = recall_score(labels.cpu(), preds.cpu(), average="binary")
        f1 = f1_score(labels.cpu(), preds.cpu(), average="binary")

        # NOTE: self.log determines the MLflow evaluation metrics
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", acc, prog_bar=True)
        self.log("train_precision", precision, prog_bar=True)
        self.log("train_recall", recall, prog_bar=True)
        self.log("train_f1", f1, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx): # NOTE: this has to be defined
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["label"]

        logits = self(input_ids, attention_mask)
        loss = F.cross_entropy(logits, labels)

        preds = torch.argmax(logits, dim=1)
        acc = (preds == labels).float().mean()
        precision = precision_score(labels.cpu(), preds.cpu(), average="binary")
        recall = recall_score(labels.cpu(), preds.cpu(), average="binary")
        f1 = f1_score(labels.cpu(), preds.cpu(), average="binary")

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", acc, prog_bar=True)
        self.log("val_precision", precision, prog_bar=True)
        self.log("val_recall", recall, prog_bar=True)
        self.log("val_f1", f1, prog_bar=True)

    def configure_optimizers(self): # NOTE: this has to be defined
        return torch.optim.AdamW(self.parameters(), lr=self.lr)


### MLflow Setup

In [3]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
MLFLOW_SERVICE_URI = os.getenv("MLFLOW_SERVICE_URI", "")

In [4]:
# setup MLflow logger and callbacks
from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, Timer
RUN_NAME = "bert_lr_5e-5_batch32_test4"

mlflow_logger = MLFlowLogger(
    experiment_name="nlp-test-exp",
    run_name=RUN_NAME,
    tracking_uri=MLFLOW_SERVICE_URI,  # can also use http://... for remote
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_f1",  # or "val_f1"
    mode="max",           # or "max" if you're monitoring accuracy/F1
    save_top_k=1,
    save_weights_only=True,
    dirpath="test_checkpoints/",
    filename=f"{RUN_NAME}-best-checkpoint-{{epoch:02d}}-{{val_f1:.2f}}",
    verbose=True
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    mode="min",
    verbose=True
)

timer = Timer()


### Lightning Trainer Setup

In [5]:
# Use the same dataset for train/val in this toy example
train_loader = DataLoader(dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(dataset, batch_size=2)

model = LitTextClassifier()

trainer = Trainer(
    max_epochs=5, 
    accelerator="cpu", # "gpu" if gpu available
    # devices=1, # if gpu available
    logger=mlflow_logger,
    log_every_n_steps=1,
    callbacks=[
        checkpoint_callback,
        early_stopping,
        timer,
    ]
)


/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
GPU available: True (cuda), used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:177: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.


### Train Model (Fit Lightning Trainer)

In [6]:
trainer.fit(model, train_loader, val_loader)

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /media/emma/10TB/home/bilab_archive/Bai/DPRecSys/sample_code/test_checkpoints exists and is not empty.

  | Name        | Type      | Params | Mode 
--------------------------------------------------
0 | transformer | BertModel | 109 M  | eval 
1 | classifier  | Linear    | 1.5 K  | train
--------------------------------------------------
109 M     Trainable params
0         Non-trainable params
109 M     Total params
437.935   Total estimated model params size (MB)
1         Modules in train mode
228       Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

Training: |          | 0/? [00:00<?, ?it/s]

/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/media/emma/10TB/home/bilab_archive/Bai/DPRecSys/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


...

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved by 0.083 >= min_delta = 0.0. New best score: 0.337
Epoch 3, global step 8: 'val_f1' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved by 0.087 >= min_delta = 0.0. New best score: 0.250
Epoch 4, global step 10: 'val_f1' was not in top 1
`Trainer.fit` stopped: `max_epochs=5` reached.


🏃 View run bert_lr_5e-5_batch32_test4 at: http://140.112.106.216:3683/#/experiments/1/runs/d4ccd7675a574e9b9f54480d3ac627b6
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/1


### Inference

In [7]:
model.eval()

sample_text = "What a fantastic experience!"
encoding = tokenizer(sample_text, return_tensors="pt", padding="max_length", truncation=True, max_length=128)

with torch.no_grad():
    logits = model(encoding["input_ids"], encoding["attention_mask"])
    pred = torch.argmax(logits, dim=1).item()
    print("Predicted Label:", pred)  # 0 or 1


Predicted Label: 1
